In [1]:
import copy
import random

import numpy as np
import torch
import torch.optim as optim

# Neural Net and Replay Buffer
from dqn import DQN, ReplayBuffer, optimize_dqn, select_epsilon_greedy_action


#### Training Loop

In [2]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "code").exists():
    project_root = project_root.parent

if not (project_root / "code").exists():
    raise RuntimeError("Could not locate the project root containing the 'code' directory.")

source_root = str(project_root / "code")
if source_root not in sys.path:
    sys.path.insert(0, source_root)

from State.traffic_env import TrafficEnv


seed = 19
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

env = TrafficEnv(max_steps=120)

state_dim = len(env._get_state())
action_dim = len(env.actions)

print(f"State size: {state_dim} values")
print(f"Action size: {action_dim} phase-duration choices")
model = DQN(state_dim, action_dim)
target_model = copy.deepcopy(model)
target_model.eval()

optimizer = optim.Adam(model.parameters(), lr=0.001)
buffer = ReplayBuffer(capacity=20000)

gamma = 0.99
epsilon = 1.0
epsilon_decay = 0.995
epsilon_min = 0.05
batch_size = 64
target_update_episodes = 25

episodes = 1000
episode_history = []

for episode in range(episodes):
    state = env.reset()
    total_reward = 0.0
    total_cars_passed = 0
    total_overtime = 0
    total_duration = 0
    losses = []
    phase_counts = {phase: 0 for phase in env.phases}

    for _ in range(env.max_steps):
        action = select_epsilon_greedy_action(model, state, action_dim, epsilon)
        action_info = env.actions[action]
        phase_counts[env.phases[action_info["phase_index"]]] += 1
        next_state, reward, done = env.step(action)

        buffer.push((state, action, reward, next_state, done))
        loss = optimize_dqn(model, target_model, optimizer, buffer, batch_size, gamma)
        if loss is not None:
            losses.append(loss)

        total_reward += reward
        total_cars_passed += env.last_step_metrics["cars_passed"]
        total_overtime += env.last_step_metrics["overtime_seconds"]
        total_duration += env.last_step_metrics["selected_duration_seconds"]
        state = next_state

        if done:
            break

    epsilon = max(epsilon_min, epsilon * epsilon_decay)
    if episode % target_update_episodes == 0:
        target_model.load_state_dict(model.state_dict())

    episode_history.append({
        "episode": episode,
        "reward": total_reward,
        "cars_passed": total_cars_passed,
        "overtime_seconds": total_overtime,
        "selected_green_seconds": total_duration,
        "phase_counts": phase_counts.copy(),
        "epsilon": epsilon,
        "loss": float(np.mean(losses)) if losses else None,
    })

    if episode % 25 == 0 or episode == episodes - 1:
        print(
            f"Episode {episode:04d} | reward {total_reward:8.2f} | "
            f"passed {total_cars_passed:4d} | green {total_duration:5.0f}s | overtime {total_overtime:4.0f}s | "
            f"left {phase_counts.get('AC_left', 0) + phase_counts.get('BD_left', 0):3d} | "
            f"epsilon {epsilon:.3f}"
        )


State size: 27 values
Action size: 45 phase-duration choices
Episode 0000 | reward -37802.60 | passed  411 | green  1800s | overtime  200s | left  17 | epsilon 0.995
Episode 0025 | reward -33245.48 | passed  451 | green  1800s | overtime  240s | left  19 | epsilon 0.878
Episode 0050 | reward -37572.41 | passed  309 | green  1810s | overtime  230s | left  11 | epsilon 0.774
Episode 0075 | reward -33472.00 | passed  411 | green  1800s | overtime  170s | left  17 | epsilon 0.683
Episode 0100 | reward -34095.59 | passed  430 | green  1810s | overtime  200s | left  19 | epsilon 0.603
Episode 0125 | reward -50376.77 | passed  417 | green  1800s | overtime  270s | left   4 | epsilon 0.532
Episode 0150 | reward -60036.50 | passed  472 | green  1800s | overtime  150s | left  20 | epsilon 0.469
Episode 0175 | reward -29671.85 | passed  370 | green  1810s | overtime  260s | left   8 | epsilon 0.414
Episode 0200 | reward -38627.03 | passed  520 | green  1850s | overtime  210s | left  32 | epsilon 

#### Save Model

In [4]:
from pathlib import Path

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "code").exists():
    project_root = project_root.parent

if not (project_root / "code").exists():
    raise RuntimeError("Could not locate the project root containing the 'code' directory.")

model_path = project_root / "code" / "Neural_Networks" / "DQN_Implementation" / "traffic_dqn_model1000.pth"
torch.save({
    "model_state_dict": model.state_dict(),
    "state_dim": state_dim,
    "action_dim": action_dim,
    "phases": env.phases,
    "actions": env.actions,
}, model_path)
print(f"Saved model checkpoint to {model_path}")


Saved model checkpoint to C:\Users\ahmad\OneDrive\Desktop\Direct\Sping2026\CPE_FinalProject\Neural-Network-Application-in-Traffic-Management-CPE-593-WS-\code\Neural_Networks\DQN_Implementation\traffic_dqn_model1000.pth
